# Demo: One-Day Dataset Generation (Tracking + Representation + Labeling)

This notebook demonstrates end-to-end dataset generation for **one trading day** from:
`data/raw/XNAS_ITCH_AAPL_mbo_20251201_20260101.dbn.zst`

Pipeline shown in this notebook:
1. Stream DBN MBO messages for a target day/session
2. Track virtual orders
3. Attach entry LOB representation
4. Apply competing-risks labels inline
5. Write a single output parquet dataset

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.order_tracking import OrderTracker

dbn_path = PROJECT_ROOT / 'data' / 'raw' / 'XNAS_ITCH_AAPL_mbo_20251201_20260101.dbn.zst'
output_path = PROJECT_ROOT / 'data' / 'datasets' / 'demo_one_day_2025-12-01.parquet'

In [2]:
output_path.parent.mkdir(parents=True, exist_ok=True)

tracker = OrderTracker(
    samples_per_day=100,
    time_censor_s=300.0,
)

tracker.process_stream(
    file_path=str(dbn_path),
    output_parquet=str(output_path),
    limit=None,
    samples_per_day=100,
    target_day='2025-12-01',
)

print({
    'scheduled_virtual': tracker.inst.scheduled_virtual,
    'spawned_virtual': tracker.inst.spawned_virtual,
})

Processed 100000 messages — last_ts=1764599474038618111, active_virtual=0
Processed 200000 messages — last_ts=1764599810979255041, active_virtual=0
Processed 300000 messages — last_ts=1764600149045829423, active_virtual=0
Processed 400000 messages — last_ts=1764600487210622894, active_virtual=0
Processed 500000 messages — last_ts=1764600815002364656, active_virtual=0
Processed 600000 messages — last_ts=1764601275023374114, active_virtual=0
Processed 700000 messages — last_ts=1764601662811990858, active_virtual=0
Processed 800000 messages — last_ts=1764602150614252948, active_virtual=0
Processed 900000 messages — last_ts=1764602597921527915, active_virtual=0
Processed 1000000 messages — last_ts=1764603062119392717, active_virtual=0
Processed 1100000 messages — last_ts=1764603678064341816, active_virtual=0
Processed 1200000 messages — last_ts=1764604435479626303, active_virtual=0
Processed 1300000 messages — last_ts=1764605178066146106, active_virtual=0
Processed 1400000 messages — last_

In [11]:
df = pd.read_parquet(output_path)
print('shape:', df.shape)
print('columns:', list(df.columns))

print('\nEvent type distribution:')
print(df['event_type'].value_counts(dropna=False).sort_index())

display(df.head(10))

shape: (100, 19)
columns: ['order_id', 'entry_time', 'duration_s', 'event', 'status_reason', 'price', 'side', 'volume', 'order_type', 'best_bid_at_entry', 'best_ask_at_entry', 'best_bid_at_post_trade', 'best_ask_at_post_trade', 'entry_representation', 'event_type', 'event_time_bin', 'post_trade_adverse_move_bps', 'post_trade_spread_bps', 'post_trade_recorded']

Event type distribution:
event_type
1    94
2     6
Name: count, dtype: int64


,order_id,entry_time,duration_s,event,status_reason,price,side,volume,order_type,best_bid_at_entry,best_ask_at_entry,best_bid_at_post_trade,best_ask_at_post_trade,entry_representation,event_type,event_time_bin,post_trade_adverse_move_bps,post_trade_spread_bps,post_trade_recorded
0,1,1764600589637787392,2.106873,1,FILLED,276760000000,A,1,VIRTUAL,276720000000,276760000000,276730000000,276750000000,"[-2762.0, -2762.0, -2758.0, -2639.0, -2334.0, ...",1,10,-1.083972,0.722700,True
1,0,1764600589637787392,12.079766,1,FILLED,276720000000,B,1,VIRTUAL,276720000000,276760000000,276850000000,276880000000,"[-2762.0, -2762.0, -2758.0, -2639.0, -2334.0, ...",1,13,-5.782018,1.083561,True
2,3,1764601051368755456,0.606819,1,FILLED,277490000000,A,1,VIRTUAL,277450000000,277490000000,277490000000,277520000000,"[-3308.0, -3183.0, -2983.0, -2872.0, -2298.0, ...",1,7,0.000000,1.081062,True
3,2,1764601051368755456,14.375897,1,FILLED,277450000000,B,1,VIRTUAL,277450000000,277490000000,277580000000,277600000000,"[-3308.0, -3183.0, -2983.0, -2872.0, -2298.0, ...",1,14,-5.406380,0.720487,True
4,5,1764601788988108544,2.180430,1,FILLED,277110000000,A,1,VIRTUAL,277070000000,277110000000,277090000000,277120000000,"[-3521.0, -3421.0, -3316.0, -3072.0, -2967.0, ...",1,10,-0.721735,1.082622,True
5,4,1764601788988108544,4.653964,1,FILLED,277070000000,B,1,VIRTUAL,277070000000,277110000000,277070000000,277080000000,"[-3521.0, -3421.0, -3316.0, -3072.0, -2967.0, ...",1,11,-0.360920,0.360913,True
6,6,1764601899766155008,0.302611,1,FILLED,277150000000,B,1,VIRTUAL,277150000000,277170000000,277140000000,277170000000,"[-4151.0, -4049.0, -3929.0, -3623.0, -3259.0, ...",1,6,-0.721631,1.082427,True
7,7,1764601899766155008,0.854751,1,FILLED,277170000000,A,1,VIRTUAL,277150000000,277170000000,277180000000,277200000000,"[-4151.0, -4049.0, -3929.0, -3623.0, -3259.0, ...",2,8,0.360789,0.721527,True
8,9,1764601980810062336,0.187220,1,FILLED,277510000000,A,1,VIRTUAL,277490000000,277510000000,277490000000,277500000000,"[-4111.0, -3998.0, -3803.0, -3703.0, -3643.0, ...",1,5,-0.720695,0.360367,True
9,8,1764601980810062336,0.224099,1,FILLED,277490000000,B,1,VIRTUAL,277490000000,277510000000,277490000000,277500000000,"[-4111.0, -3998.0, -3803.0, -3703.0, -3643.0, ...",1,6,-0.360373,0.360367,True


In [4]:
sample_vec = df['entry_representation'].iloc[0]
print('entry_representation length:', len(sample_vec) if sample_vec is not None else None)
print('first 10 values:', sample_vec[:10] if sample_vec is not None else None)

entry_representation length: 41
first 10 values: [-2762. -2762. -2758. -2639. -2334. -2329. -1924. -1769. -1669. -1540.]
